<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/3%20-%20Regress%C3%A3o%20Log%C3%ADstica/3.3_datasets_e_dataloaders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 3.3 — Datasets e DataLoaders no PyTorch

## Objetivos

- Criar um dataset a partir de tensores com `torch.utils.data.TensorDataset`.
- Criar um dataset próprio herdando de `torch.utils.data.Dataset` (métodos `__len__` e `__getitem__`).
- Dividir os dados em mini-batches com `torch.utils.data.DataLoader`, usando `batch_size` e `shuffle`.
- Distinguir época, iteração e mini-batch.

## O problema

No notebook 3.2 usamos 1000 imagens do MNIST de uma vez em cada época (batch completo). Com as 60 mil imagens, cada passo do gradiente descendente fica caro, e em problemas maiores os dados nem cabem na memória. A saída é dividir os dados em **mini-batches**: pequenos grupos de amostras, cada um usado para uma atualização dos parâmetros. O notebook 3.4 treina o MNIST assim.

Antes, vamos ver as ferramentas do PyTorch para isso com um exemplo pequeno, que cabe num gráfico: as 50 flores versicolor do Iris, com o comprimento da sépala como entrada e o comprimento da pétala como saída (um problema de regressão, como no módulo 2). O PyTorch oferece:

* `torch.utils.data.TensorDataset` e `torch.utils.data.Dataset`: duas formas de criar um **dataset**, o objeto que guarda e indexa as amostras;
* `torch.utils.data.DataLoader`: percorre um dataset em mini-batches.

## Importação das bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.utils.data import TensorDataset, Dataset, DataLoader

from sklearn.datasets import load_iris

_ = torch.manual_seed(0)  # fixa também a ordem do embaralhamento (shuffle) do DataLoader

## Datasets

No PyTorch, datasets são objetos usados para armazenar, indexar, transformar e retornar as amostras de um conjunto de dados. Existem duas formas de criar um dataset:

* transformar os dados em tensores e criar o dataset com `torch.utils.data.TensorDataset`;
* criar uma classe filha de `torch.utils.data.Dataset`.

### Criando um dataset com `TensorDataset`

Se os dados e os rótulos estão em arrays NumPy ou em tensores, é possível criar um dataset com [torch.utils.data.TensorDataset](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.TensorDataset).

#### Dados anotados: `x_data` (comprimento das sépalas) e `target` (comprimento das pétalas)

In [ ]:
iris = load_iris()
data = iris.data[iris.target == 1, ::2]  # só versicolor (classe 1); colunas 0 e 2: comprimento da sépala e da pétala (cm)

x_data = data[:, 0:1]  # 0:1 em vez de 0 mantém a segunda dimensão: shape (50, 1)
target = data[:, 1:2]

print('dimensões de x_data:', x_data.shape)
print('dimensões de target:', target.shape)

#### Passo 1: transformar os arrays NumPy em tensores

In [ ]:
x_data = torch.tensor(x_data, dtype=torch.float32)
target = torch.tensor(target, dtype=torch.float32)

#### Passo 2: usar `TensorDataset` para criar o dataset com os tensores

In [ ]:
dataset = TensorDataset(x_data, target)
len(dataset)

O dataset pode ser indexado como uma lista, e cada elemento é uma tupla com:
- o dado de entrada e
- o dado de saída.

In [ ]:
i = 10
(x, y) = dataset[i]

print('sépala:', x)
print('pétala:', y)

### Criando um dataset com `torch.utils.data.Dataset`

Para casos mais complexos é possível criar uma classe que representa o dataset. Essa classe herda de [torch.utils.data.Dataset](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.Dataset).

#### Estrutura do dataset

Quem herda de `Dataset` precisa implementar dois métodos:
- `__len__`: retorna o tamanho do dataset;
- `__getitem__`: retorna o elemento de um índice dado.

O método `__len__` é chamado quando usamos `len(dataset)` e `__getitem__` é chamado quando fazemos a indexação `dataset[i]`.

O construtor (`__init__`) recebe os dados e pode prepará-los. Aqui ele converte os arrays NumPy em tensores `float32`, o tipo que os modelos do PyTorch esperam.

In [ ]:
class IrisDataset(Dataset):
    def __init__(self, x_data, target):
        super().__init__()

        self.data = torch.tensor(x_data, dtype=torch.float32)
        self.target = torch.tensor(target, dtype=torch.float32)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, index):

        sepala = self.data[index]
        petala = self.target[index]

        return (sepala, petala)

Nossa classe está pronta; podemos criar um objeto dela.

#### Criando e testando um objeto da nossa classe

Desta vez passamos os dados como arrays NumPy; a conversão para tensor acontece dentro da classe.

In [ ]:
x = data[:, 0:1]
y = data[:, 1:2]

dataset = IrisDataset(x, y)

O método `__len__` deve retornar o tamanho da primeira dimensão dos dados (o número de amostras). Podemos verificar chamando `len()`:

In [ ]:
print(len(dataset))

O método `__getitem__` deve retornar uma tupla com a amostra da posição `i`, no nosso caso `data[i]` e `target[i]`:

In [ ]:
i = 1
print(dataset[i])

In [ ]:
data_i, target_i = dataset[30]

print('sépala:', data_i)
print('pétala:', target_i)

### Quando usar cada forma

O exemplo acima é simples e serve para mostrar a estrutura que o PyTorch usa para representar datasets. Uma classe filha de `Dataset` é útil quando os dados não cabem num único tensor ou precisam de preparação: ler imagens de arquivos sob demanda, aplicar uma transformação em cada amostra, etc. O `torchvision.datasets.MNIST`, usado nos notebooks 3.2 e 3.4, é uma classe desse tipo.

## DataLoaders

Um DataLoader ([torch.utils.data.DataLoader](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)) percorre um dataset e entrega as amostras em mini-batches: a cada passo do `for` ele devolve um novo grupo de `batch_size` amostras, já empilhadas em tensores. Com `shuffle=True`, a ordem das amostras é sorteada de novo a cada passada completa pelo dataset.

O DataLoader é a ferramenta do PyTorch para o gradiente descendente por mini-batches, que tem duas vantagens:
- cada atualização dos parâmetros usa só um mini-batch, então fazemos muitas atualizações por passada pelos dados (gradiente descendente estocástico por mini-batch);
- só os dados de um mini-batch precisam estar na memória (normalmente da GPU) de cada vez.

Três termos que costumam se misturar:
- **mini-batch**: um grupo de `batch_size` amostras;
- **iteração**: uma atualização dos parâmetros, feita com um mini-batch;
- **época (epoch)**: uma passada completa por todas as amostras do dataset, ou seja, `len(data_loader)` iterações.

### Criando um DataLoader

Vamos utilizar o dataset criado anteriormente como fonte de dados do DataLoader.

In [ ]:
print('tamanho do dataset: ', len(dataset))
print('amostra 12:', dataset[12])

Agora podemos criar o DataLoader com o dataset:

In [ ]:
data_loader = DataLoader(dataset,
                         batch_size=10,  # tamanho do mini-batch
                         shuffle=False)  # se True, embaralha os dados no início de cada época

### Iterando sobre o DataLoader

Podemos iterar sobre o DataLoader com um `for`. Cada passada completa pelo `for` é uma época.

In [ ]:
batch_n = 0
n_samples = 0
for data_batch in data_loader:
    print('batch ', batch_n)
    batch_n += 1
    n_samples += len(data_batch[0])

print('tamanho do DataLoader', len(data_loader))
print('tamanho do dataset', n_samples)

O tamanho do DataLoader é o número de mini-batches: 50 amostras em mini-batches de 10 dão 5 mini-batches por época.

Agora podemos utilizar os dados do DataLoader:

In [ ]:
batch_n = 0

for data_batch, targets_batch in data_loader:  # cada mini-batch é a tupla (sépalas, pétalas)
    print('dimensão do batch de sépalas {}: {}'.format(batch_n, data_batch.size()))
    print('dimensão do batch de pétalas {}: {}\n'.format(batch_n, targets_batch.size()))
    batch_n += 1

Cada mini-batch tem 10 amostras: 5 tensores 10 × 1 de sépalas e 5 tensores 10 × 1 de pétalas. O DataLoader empilhou as amostras de shape (1,) devolvidas por `__getitem__` num tensor (10, 1).

### Efeito do `shuffle`

Nos gráficos abaixo, cada cor e marcador mostra as amostras de um mini-batch, em duas épocas seguidas. Com `shuffle=False`, cada mini-batch tem sempre as mesmas amostras.

In [ ]:
def plot_two_epochs(data_loader):
    color = ["b", "g", "r", "c", "m"]
    mark = ['+', '.', '^', 'o', 'x']
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, title in zip(axes, ['Primeira época', 'Segunda época']):
        for i, (data_batch, targets_batch) in enumerate(data_loader):
            ax.scatter(data_batch, targets_batch, c=color[i], marker=mark[i], label=f'mini-batch {i}')
        ax.set_xlabel('comprimento da sépala (cm)')
        ax.set_ylabel('comprimento da pétala (cm)')
        ax.set_title(title)
        ax.legend()
    plt.show()

In [ ]:
data_loader = DataLoader(dataset,
                         batch_size=10,  # tamanho do mini-batch
                         shuffle=False)  # se True, embaralha os dados no início de cada época
plot_two_epochs(data_loader)

Com `shuffle=True`, a composição dos mini-batches muda a cada época:

In [ ]:
data_loader = DataLoader(dataset,
                         batch_size=10,  # tamanho do mini-batch
                         shuffle=True)   # embaralha os dados no início de cada época
plot_two_epochs(data_loader)

No treinamento usamos `shuffle=True` nos dados de treino: assim os mini-batches não seguem sempre a mesma ordem (por exemplo, ordenada por classe), e cada época vê combinações diferentes de amostras.

## Exercícios

1. Defina seu próprio DataLoader na célula abaixo, com mini-batches de 5 elementos, usando o dataset já criado. Em seguida imprima o tamanho do DataLoader, o tamanho do dataset e os valores de sépala do primeiro mini-batch.

In [ ]:
# -- Defina aqui seu DataLoader com batches de 5 elementos cada --
# TODO: escreva sua solução aqui


2. A ordem dos elementos se altera entre execuções da célula acima? Coloque o parâmetro `shuffle` do DataLoader como `True`, execute a célula várias vezes e observe o resultado.

*Escreva sua resposta aqui.*

3. Crie um DataLoader com `batch_size=8`. Quantos mini-batches ele tem e qual é o tamanho de cada um? O que muda com o argumento `drop_last=True`?

In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

## Aprendizados

- Um dataset do PyTorch responde a `len(dataset)` e `dataset[i]`; `TensorDataset` resolve o caso em que os dados já estão em tensores, e uma classe filha de `Dataset` (com `__len__` e `__getitem__`) resolve os demais.
- O DataLoader entrega os dados em mini-batches de `batch_size` amostras; `len(data_loader)` é o número de mini-batches por época.
- Época, iteração e mini-batch são coisas diferentes: uma época tem `len(data_loader)` iterações, uma por mini-batch.
- `shuffle=True` sorteia uma nova ordem a cada época; é o padrão para os dados de treino.